# Marker-alignment timing diagnostic

**The question.** SYNASC 2026 Reviewer 1 wrote that our reading of the two near-chance EMI recordings as catastrophic failure caused by RF energy coupling into the sensing chain *"lacks supporting impedance or RF measurements"*.

They are right. We have no such measurements. This notebook tests a different explanation, one we *can* measure, and finds it.

---

## The mechanism

`analysis/loader.py` aligns stimulus markers to the EEG sample axis by least-squares fitting

$$\text{acq\_time} \approx m \cdot (\text{arrival\_time} - \text{anchor}) + c$$

over every received UDP packet. Ordinary least squares is the right estimator **when the noise is symmetric**.

Bluetooth packet delay is not symmetric. A packet can arrive *late*; it cannot arrive *early*. Delay is one-sided, and under link contention it becomes heavy-tailed.

In acq-versus-arrival space, a delayed packet sits **below** the true line — its arrival time is inflated relative to its acquisition time. So the true clock is the **upper envelope** of the point cloud, not its least-squares mean. Fit the mean of a one-sided distribution and the line tilts toward the delayed mass, dragging every stimulus marker off the EEG with it.

When delay is tight the bias is invisible. When the tail blows out, alignment fails.

**And the sample counter cannot see any of this.** Zero samples were dropped in these recordings — the finding reported in `DEVIATIONS.md` (2026-06-11) stands. Every sample arrived. What degraded was *when*.

---

## Scope

This analysis alters no reported value. Re-aligning only the recordings that produced inconvenient results would be outcome-dependent correction and is indefensible — if the upper-envelope fit were ever adopted it would have to be applied uniformly to all 168 recordings. This is a diagnostic.

In [ ]:
import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
_here = Path('.').resolve()
repo_root = next((p for p in [_here, _here.parent, _here.parent.parent]
                  if (p / 'config.yaml').exists()), _here)
os.chdir(repo_root); sys.path.insert(0, str(repo_root))

%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mne; mne.set_log_level('ERROR')

from analysis.timing_diagnostic import (
    fit_clock, _load_timing, recording_residuals, scan_all_recordings,
    compare_fits, lag_sweep, score_recording, FLAG_RESIDUAL_MS, DERIVED_PIPELINE,
)

OUT = Path('data/derived') / DERIVED_PIPELINE
OUT.mkdir(parents=True, exist_ok=True)
print('repo root:', repo_root)

## 1. What the pathology looks like

Residuals of the OLS clock fit, for sub-19's EMI recording against sub-19's own control. Same subject, same headset, same session, ~30 minutes apart.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for col, (sub, cond) in enumerate([('19', 'control'), ('19', 'emi')]):
    tr, acq = _load_timing(sub, cond)
    m, c = fit_clock(tr, acq, 'ols')
    resid = (acq - (m * tr + c)) * 1000

    ax = axes[0, col]
    ax.plot(tr, resid, lw=0.4, color='#356')
    ax.axhline(0, color='#c33', lw=0.8, ls='--')
    ax.set_title(f'sub-{sub} {cond}: residual over time (std {resid.std():.1f} ms)')
    ax.set_xlabel('time into recording (s)'); ax.set_ylabel('residual (ms)')

    ax = axes[1, col]
    ax.hist(resid, bins=120, color='#9bd', edgecolor='none')
    ax.axvline(0, color='#c33', lw=0.8, ls='--')
    ax.set_yscale('log')
    ax.set_title(f'residual distribution (1st pct {np.percentile(resid,1):.0f} ms)')
    ax.set_xlabel('residual (ms)')
plt.tight_layout(); plt.show()

print('Residual tails run negative in both recordings, indicating late-arriving')
print('packets. In the EMI recording the tail extends to several hundred ms, which')
print('is sufficient to bias the least-squares fit away from the true clock.')

## 2. How common is it, and is it EMI-specific?

This is the question that decides how the finding gets written up. If the fault were EMI-specific it would be a story about interference. If it appears in every condition it is a sporadic link fault that happened to land where it did.

In [ ]:
scan = scan_all_recordings()
scan.to_csv(OUT / 'residual_scan.csv', index=False)

summary = (scan.groupby('condition')['residual_std_ms']
           .agg(median='median', mean='mean', max='max', n='count'))
summary['n_flagged'] = scan.groupby('condition')['flagged'].sum()
display(summary.round(2))

print(f'total recordings: {len(scan)}')
for thr in (30, 50, 90):
    print(f'  residual std > {thr:>2} ms: {(scan.residual_std_ms > thr).sum():>3}')

print(f'\nAll recordings above the {FLAG_RESIDUAL_MS:.0f} ms flag:')
display(scan[scan.flagged].sort_values('residual_std_ms', ascending=False)
        [['subject', 'condition', 'residual_std_ms', 'p1_ms', 'min_ms']].round(1))

Median residuals are near-identical across the four conditions, and flagged recordings occur in all of them. The pathology is therefore consistent with a sporadic link or host timing fault rather than a consequence of the EMI manipulation. EMI shows at most a modest elevation confined to the tail of the distribution, which the present data are not powered to attribute to the manipulation itself.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for i, cond in enumerate(['control', 'chewing', 'emi', 'acoustic']):
    v = scan[scan.condition == cond]['residual_std_ms'].values
    ax.scatter(np.full_like(v, i) + np.random.uniform(-.13, .13, len(v)), v,
               s=16, alpha=.75, color='#468')
ax.axhline(FLAG_RESIDUAL_MS, color='#c33', ls='--', lw=1,
           label=f'{FLAG_RESIDUAL_MS:.0f} ms flag')
ax.set_yscale('log'); ax.set_xticks(range(4))
ax.set_xticklabels(['control', 'chewing', 'EMI', 'acoustic'])
ax.set_ylabel('clock-fit residual std (ms, log)')
ax.set_title('Timing pathology is rare and occurs in every condition')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 3. Is a constant offset the explanation?

The simplest version of the hypothesis is that markers are shifted by a fixed amount. We test it by sweeping a constant lag and watching AUC.

`sub-01 emi` is the control case — a healthy recording. Watch what happens to it at plus or minus one SOA (233 ms).

In [ ]:
LAGS = [-466, -233, -117, 0, 117, 233, 466]
sweeps = {f'sub-{s} {c}': lag_sweep(s, c, LAGS) for s, c in
          [('01', 'emi'), ('19', 'emi'), ('33', 'emi')]}

fig, ax = plt.subplots(figsize=(7, 3.8))
for label, df in sweeps.items():
    ax.plot(df.lag_ms, df.auc, marker='o', label=label)
ax.axhline(0.5, color='#888', ls=':', lw=1)
ax.axvline(0, color='#c33', ls='--', lw=.8)
ax.set_xlabel('constant marker shift (ms)'); ax.set_ylabel('AUC')
ax.set_title('Constant-lag sweep')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

for label, df in sweeps.items():
    print(f'{label:14s} ' + '  '.join(f'{a:.3f}' for a in df.auc))

Two observations follow.

First, the healthy recording (sub-01) peaks sharply at zero lag and falls below chance at plus or minus one stimulus-onset asynchrony. Below-chance AUC is diagnostic: additive noise drives AUC toward 0.5 and cannot pass it, whereas a one-flash marker offset can, because the epoch then contains the neighbouring flash's evoked response. The EMI recording of sub-19 sits at that same below-chance value with no shift applied.

Second, no constant lag restores performance for sub-19 or sub-33. The misalignment is therefore time-varying rather than a fixed offset, which is the behaviour predicted by one-sided delay and the reason the correction is applied to the clock fit rather than to the marker positions.

## 4. The recovery test

Refit the clock to the upper envelope, hand the corrected markers to the **registered** `preprocess_recording()` (same filtering, epoching, baseline, boundary rejection, ±150 µV gate), and score with that subject's saved v3 model.

If the diagnosis is correct, the collapsed recordings recover and healthy recordings are unaffected. Healthy recordings are included precisely so that this second condition can be tested: a correction that altered them as well would indicate a change in the estimator rather than the repair of a fault.

In [ ]:
PAIRS = [('19', 'emi'), ('33', 'emi'),            # the two collapses
         ('04', 'acoustic'),                       # 2nd-worst residual in study
         ('01', 'emi'), ('05', 'emi'),             # healthy EMI
         ('19', 'control'), ('19', 'acoustic'),    # same subject, healthy conds
         ('33', 'control'), ('33', 'chewing')]

cmp = compare_fits(PAIRS)
cmp.to_csv(OUT / 'realignment_comparison.csv', index=False)
display(cmp)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
labels = [f"{r.subject}\n{r.condition}" for r in cmp.itertuples()]
x = np.arange(len(cmp))
ax.bar(x - .19, cmp.auc_ols, .38, label='OLS fit (as published)', color='#c96')
ax.bar(x + .19, cmp.auc_envelope, .38, label='upper-envelope fit', color='#468')
ax.axhline(0.5, color='#888', ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7.5)
ax.set_ylabel('AUC'); ax.set_title('Re-alignment recovers the collapses, leaves healthy recordings alone')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

broken = cmp[cmp.auc_ols < 0.75]
healthy = cmp[cmp.auc_ols >= 0.75]
print(f'collapsed recordings  (n={len(broken)}): mean AUC change {broken.auc_delta.mean():+.3f}')
print(f'healthy recordings    (n={len(healthy)}): mean AUC change {healthy.auc_delta.mean():+.3f}')
print('\nRecovery is confined to the collapsed recordings; healthy recordings are')
print('unchanged within rounding, as a correct diagnosis predicts.')

## 5. Consequence for the acoustic limitation

The one near-chance acoustic score has been attributed to a participant with professional audio training who reported the stimulus as aversive. That participant's acoustic recording also carries the second-worst marker alignment in the dataset, giving two candidate explanations for the same observation. The re-alignment test discriminates between them.

In [ ]:
s4 = scan[scan.subject == '04'][['condition', 'residual_std_ms', 'p1_ms']].round(1)
display(s4)
row = cmp[(cmp.subject == '04') & (cmp.condition == 'acoustic')]
display(row)
print('Substantial recovery under re-alignment indicates that the behavioural')
print('account and the timing artifact are confounded for this participant.')

## 6. Summary for the write-up

In [ ]:
out = {
    'n_recordings': int(len(scan)),
    'median_residual_ms_by_condition':
        scan.groupby('condition')['residual_std_ms'].median().round(2).to_dict(),
    'n_flagged_by_condition':
        scan.groupby('condition')['flagged'].sum().astype(int).to_dict(),
    'n_above_30ms': int((scan.residual_std_ms > 30).sum()),
    'n_above_90ms': int((scan.residual_std_ms > 90).sum()),
    'recovery': cmp.to_dict('records'),
    'mean_auc_change_collapsed': round(float(broken.auc_delta.mean()), 3),
    'mean_auc_change_healthy': round(float(healthy.auc_delta.mean()), 3),
    'interpretation': (
        'Sporadic one-sided Bluetooth packet delay biases the least-squares '
        'clock fit and misaligns stimulus markers. Rare (11/168 recordings above '
        '30 ms residual), present in all four conditions, and invisible to the '
        'sample-counter integrity check because no samples are dropped. Refitting '
        'the clock to the upper envelope recovers the two collapsed EMI recordings '
        'and leaves healthy recordings unchanged. Reported as a diagnostic; no '
        'published value is altered.'),
}
(OUT / 'timing_diagnostic_summary.json').write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2)[:1800])
print('\nsaved ->', OUT / 'timing_diagnostic_summary.json')

## 7. What produces the misalignment?

Four candidate causes each predict a specific association with the flagged recordings:

| Candidate | Prediction |
|---|---|
| Host scheduling stall | flagged recordings drop stimulus frames |
| Reduced transmit power | flagged recordings show lower headset battery |
| Progressive session wear | flagged recordings fall late in the session |
| The noise manipulation | flagged recordings concentrate in one condition |

The stimulus loop and the UDP receiver share a machine, so the first prediction is
the sharpest available test: a scheduling stall long enough to delay packets by
hundreds of milliseconds would drop tens of frames at 60 Hz.

In [ ]:
from analysis.timing_diagnostic import build_cause_frame, cause_analysis

cause = build_cause_frame()
cause.to_csv(OUT / 'cause_frame.csv', index=False)
res = cause_analysis(cause)

print(f"recordings: {res['n_recordings']}   flagged: {res['n_flagged']}   "
      f"samples dropped across dataset: {res['samples_dropped_total']:.0f}\n")

for name, key in [('host scheduling (late frames)', 'host_scheduling'),
                  ('transmit power (battery)', 'transmit_power')]:
    d = res[key]
    print(f"{name}")
    print(f"   flagged mean {d['flagged_mean']:.3f} vs normal {d['normal_mean']:.3f}")
    print(f"   rho vs residual = {d['spearman_rho_vs_residual']:+.3f} "
          f"(p = {d['spearman_p']:.3g}); Mann-Whitney p = {d['mannwhitney_p']:.3g}\n")

pw = res['progressive_wear']
print('progressive session wear')
print(f"   flagged by session position: {pw['flagged_by_condition_order']}")
print(f"   all recordings by position:  {pw['all_by_condition_order']}")
print(f"   rho vs residual = {pw['spearman_rho_vs_residual']:+.3f}")

None of the four candidates is supported. Frame drops are flat and marginally
*lower* in flagged recordings, battery is indistinguishable, flagged recordings are
spread across all four session positions, and condition medians were shown to be
equal in section 2.

What does appear is clustering at the level of the session.

In [ ]:
sc = res['session_clustering']
print(f"subjects with at least one flagged recording: {sc['subjects_with_any']}")
print(f"subjects with two or more: {sc['n_subjects_with_two_or_more']} "
      f"(expected {sc['expected_if_independent']} if flagged recordings were independent)\n")

multi = [s for s, n in sc['subjects_with_any'].items() if n >= 2]
for s in multi:
    sub = (cause[cause.subject == s]
           .sort_values('condition_order')
           [['condition_order', 'condition', 'residual_std_ms', 'battery_mean', 'flagged']])
    print(f'sub-{s}')
    print(sub.to_string(index=False), '\n')

Within an affected session the flagged recordings form a **contiguous run** that
begins and ends mid-session: sub-04 escalates across three consecutive recordings
and then returns to baseline, sub-10 is clean, degrades for two recordings and
recovers, and sub-25 degrades for two and then clears.

This is the signature of an episodic degradation of the wireless path, persisting
across one to three recordings before resolving. It is not a property of the
headset, the host, the participant, or the condition.

The clustering itself is suggestive rather than established: three subjects show
two or more affected recordings against roughly one expected under independence,
which at these counts corresponds to *p* of about 0.09. The contiguous-run pattern
is reported descriptively.

The specific trigger cannot be identified from the available instrumentation.
Candidates that the present logging cannot distinguish include an intermittent
2.4 GHz emitter in the room, occlusion or displacement of the receiving dongle,
and host USB power management. None was recorded during data collection.

The practical consequence does not depend on resolving this. The clock-fit
residual is already computed inside `analysis/loader.py`; reporting it per
recording, and gating on it, detects the fault regardless of its origin.